# Question 8 - 122. Best Time to Buy and Sell Stock II

You are given an integer array `prices` where `prices[i]` is the price of a given stock on the `i-th` day.

On each day, you may decide to buy and/or sell the stock. You can only hold at most one share of the stock at any time. However, you can buy it then immediately sell it on the same day.

Find and return the **maximum profit** you can achieve.

**Example 1:**

    Input: prices = [7,1,5,3,6,4]
    Output: 7
    Explanation: Buy on day 2 (price = 1) and sell on day 3 (price = 5), profit = 5-1 = 4.
    Then buy on day 4 (price = 3) and sell on day 5 (price = 6), profit = 6-3 = 3.
    Total profit is 4 + 3 = 7.

**Example 2:**

    Input: prices = [1,2,3,4,5]
    Output: 4
    Explanation: Buy on day 1 (price = 1) and sell on day 5 (price = 5), profit = 5-1 = 4.
    Total profit is 4.

**Example 3:**

    Input: prices = [7,6,4,3,1]
    Output: 0
    Explanation: There is no way to make a positive profit, so we never buy the stock to achieve the maximum profit of 0.

**Constraints:**

- `1 <= prices.length <= 3 * 10^4`
- `0 <= prices[i] <= 10^4`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy: collect every day-to-day price increase

📘 **Revise first:** [Pattern C · running best-so-far](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (try every buy/sell/skip choice) | O(2ⁿ) | O(n) |
| 1½ DP with two states (holding / not holding) | O(n) | O(1) |
| 2️⃣ Optimized (sum of positive differences) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Same stock prices as Question 7, but now you can trade **as many times as you like**. The only rule: you can hold **at most one share** at a time (sell before you buy again). Same-day sell-then-buy is allowed.

```
price:  7  1  5  3  6  4
           ↑──↑  ↑──↑
          +4     +3        → total 7
```

**Everyday picture: you have a crystal ball for tomorrow only.** Each evening you can see tomorrow's price.
- If tomorrow is **higher** → buy today, sell tomorrow, pocket the difference.
- If tomorrow is **lower or equal** → don't hold anything overnight.

That simple rule **is** optimal. Any long climb, like 1 → 3 → 5, earns the same whether you hold the whole way (5 − 1 = 4) or take each step separately ((3 − 1) + (5 − 3) = 4). The rule automatically skips every drop.

### Step 0 · Clarify before coding

🗣️ *"So I can make unlimited transactions, but hold only one share at a time, and I can sell and buy again on the same day. I return the maximum total profit."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| How many trades? | **Unlimited.** | Every rise can be captured. That's why greedy works. |
| Hold more than one share? | **No**, at most one. | Can't "stack" buys. |
| Sell and buy on the same day? | **Yes.** | Lets us split a long rise into daily pieces with no loss. |
| Fees or cooldowns? | **None.** | With fees, greedy breaks (see the follow-ups). |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** try **every possible sequence of decisions**. Each day, if you're **not holding**, you can *buy* or *skip*. If you're **holding**, you can *sell* or *skip*. Explore all of them recursively and keep the best total.

🗣️ *"The brute force explores every buy/sell/skip decision recursively. The state is which day it is and whether I'm holding a share. Every day branches into two choices, so it's exponential, about O(2ⁿ)."*

**Why is it slow?** Two choices per day across n days is 2ⁿ paths. With 30 days that's already about a billion. The waste: many paths reach the **same situation** (same day, same holding status) and recompute everything from there.

In [1]:
class SolutionBrute:
    def maxProfit(self, prices: list[int]) -> int:
        def best_from(day: int, holding: bool) -> int:
            if day == len(prices):
                return 0
            skip = best_from(day + 1, holding)
            if holding:
                act = prices[day] + best_from(day + 1, False)    # sell today
            else:
                act = -prices[day] + best_from(day + 1, True)    # buy today
            return max(skip, act)
        return best_from(0, False)

#### Step 1½ · Remember the two states (dynamic programming)

The only things that matter on any day are: *"what's my best money if I'm **not holding** now?"* (`cash`) and *"…if I **am holding** now?"* (`hold`). Each day:
- `cash = max(cash, hold + price)` → stay out, **or** sell today.
- `hold = max(hold, cash − price)` → keep holding, **or** buy today.

🗣️ *"The recursion repeats the same (day, holding) states. Keeping just two running values, best cash and best holding, turns it into O(n) time and O(1) space. This is the version that generalises to fees and cooldowns. But for this exact problem there's an even simpler view."*

In [2]:
class SolutionDP:
    def maxProfit(self, prices: list[int]) -> int:
        cash, hold = 0, float("-inf")      # not holding / holding one share
        for p in prices:
            cash, hold = max(cash, hold + p), max(hold, cash - p)
        return cash

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha: a long profitable hold is just the sum of its daily steps.**
> Buying on day `i` and selling on day `j` earns `p[j] − p[i]`, which equals
> `(p[i+1] − p[i]) + (p[i+2] − p[i+1]) + … + (p[j] − p[j−1])`.

So any strategy's profit is a **sum of some daily changes**. To make it as large as possible: **take every positive daily change and skip every negative one.** Since same-day sell-and-rebuy is free, we can always do that.

🗣️ *"Since trades are unlimited and free, any rising stretch can be split into day-by-day gains without losing anything, and I'd never want to hold through a down day, because I could sell before it and buy back after. So the maximum profit is just the sum of all positive differences between consecutive days. One pass, O(n) time, O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Try every decision sequence | ❌ | Exponential, O(2ⁿ). |
| Find each "valley then peak" and add peak − valley | ✅ | Same answer and O(n), but more index bookkeeping (walk down to a valley, then up to a peak). |
| Two-state DP (`cash` / `hold`) | ✅ **most general** | O(n)/O(1). It's the version to switch to when the interviewer adds a **fee** or **cooldown**. |
| **Sum of positive daily differences** | ✅ **simplest** | Three lines and easy to prove. Optimal for exactly this problem. |
| Reuse Question 7's "one best trade" idea | ❌ | That picks only **one** trade. Here we need many. |

In [3]:
class Solution:
    def maxProfit(self, prices: list[int]) -> int:
        profit = 0
        for i in range(1, len(prices)):
            gain = prices[i] - prices[i - 1]
            if gain > 0:            # price went up overnight: take that step
                profit += gain
        return profit

### Step 3 · Toy example walkthrough (tell it like a story)

`prices = [7, 1, 5, 3, 6, 4]`

| day i | change prices[i] − prices[i−1] | take it? | total profit |
|---|---|---|---|
| 1 | 1 − 7 = −6 | ❌ (a drop) | 0 |
| 2 | 5 − 1 = **+4** | ✅ | 4 |
| 3 | 3 − 5 = −2 | ❌ | 4 |
| 4 | 6 − 3 = **+3** | ✅ | 7 |
| 5 | 4 − 6 = −2 | ❌ | 7 |

Answer **7** ✅, the same as "buy at 1, sell at 5" plus "buy at 3, sell at 6".

**Narrated:** "From 7 to 1 it drops, so I wouldn't hold. From 1 to 5 it rises by 4, so I take it. From 5 to 3 it drops, skip. From 3 to 6 it rises by 3, take it. From 6 to 4 it drops, skip. Total 7."

**Steady climb:** `[1, 2, 3, 4, 5]` → +1 +1 +1 +1 = 4, exactly the same as one trade from 1 to 5. Splitting a climb into steps never loses money.

In [4]:
cases = [
    ([7, 1, 5, 3, 6, 4], 7),
    ([1, 2, 3, 4, 5], 4),
    ([7, 6, 4, 3, 1], 0),
    ([5], 0),
    ([2, 1, 2, 0, 1], 2),
]
for prices, expected in cases:
    for Impl in (SolutionBrute, SolutionDP, Solution):
        assert Impl().maxProfit(prices) == expected, Impl.__name__
    print(prices, "->", expected)

import random
for _ in range(300):
    p = [random.randint(0, 9) for _ in range(random.randint(1, 10))]
    assert Solution().maxProfit(p) == SolutionBrute().maxProfit(p) == SolutionDP().maxProfit(p), p
print("All tests passed ✅")

[7, 1, 5, 3, 6, 4] -> 7
[1, 2, 3, 4, 5] -> 4
[7, 6, 4, 3, 1] -> 0
[5] -> 0
[2, 1, 2, 0, 1] -> 2
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Every decision sequence | O(2ⁿ) | O(n) call stack | Doubles with every extra day. |
| Two-state DP | O(n) | O(1) | One pass, two running numbers. |
| **Sum of positive changes** | **O(n)** | **O(1)** | One pass, one running total. |

**Why O(n) time?** One loop over consecutive pairs, with one subtraction and one comparison each.

**Why O(1) memory?** Only `profit` (and the loop index) is stored.

**Edge cases to mention:** one day (0), only falling (0), flat prices (0), a steady climb (same as one big trade).

**Likely follow-ups:**
- *Transaction fee* (LC 714) → greedy breaks, because many small trades now cost more. Use the DP and subtract the fee when selling: `cash = max(cash, hold + p − fee)`.
- *Cooldown after selling* (LC 309) → add a third state, "just sold".
- *At most k trades* → Questions 148/149 (DP).

---

#### 🗣️ Full interview script (about 60 seconds)

*"Now I can trade as many times as I like, holding at most one share, and I can sell and rebuy on the same day.*

*Brute force tries every buy, sell, or skip decision, which is exponential. Recognising the repeated states gives a two-variable DP: best cash if not holding, best value if holding. That's O(n).*

*But there's a simpler view. The profit of any hold from day i to day j equals the sum of the daily changes in between. With unlimited free trades, I should collect every positive daily change and skip every negative one, because I'd never want to hold through a drop. So the answer is the sum of all positive prices[i] − prices[i−1].*

*That's one pass, O(n) time and O(1) space. If a fee or cooldown is added, I'd switch back to the state DP."*